# Apollo Phase 0 — Latency Benchmark

Test inference latency for candidate model architectures on local hardware.
Goal: determine what model size/architecture can run within <20ms per event.

Candidates:
1. Small Transformer (GPT-2 style, various sizes)
2. Mamba / State Space Model (linear-time inference)
3. Simple GRU/LSTM baseline

In [ ]:
import torch
import torch.nn as nn
import time
import numpy as np
import pandas as pd
import json
from pathlib import Path

device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# --- Model Definitions ---

class SmallTransformer(nn.Module):
    """GPT-2 style autoregressive transformer for event generation."""
    def __init__(self, vocab_size=512, d_model=256, nhead=4, num_layers=4, max_seq_len=512):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4, 
            dropout=0.0, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.output_head = nn.Linear(d_model, vocab_size)
        self.max_seq_len = max_seq_len
    
    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)
        h = self.embedding(x) + self.pos_embedding(pos)
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
        memory = torch.zeros(B, 1, self.d_model, device=x.device)
        h = self.transformer(h, memory, tgt_mask=mask)
        return self.output_head(h)


class SimpleGRU(nn.Module):
    """GRU baseline for comparison."""
    def __init__(self, vocab_size=512, d_model=256, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.gru = nn.GRU(d_model, d_model, num_layers=num_layers, batch_first=True)
        self.output_head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x, hidden=None):
        h = self.embedding(x)
        h, hidden = self.gru(h, hidden)
        return self.output_head(h), hidden


class CompoundEventTransformer(nn.Module):
    """Transformer with factored output heads for compound events.
    Predicts pitch, velocity, duration, pedal, and timbre simultaneously."""
    def __init__(self, d_model=256, nhead=4, num_layers=4, max_seq_len=256):
        super().__init__()
        self.d_model = d_model
        # Input embeddings for each field
        self.pitch_emb = nn.Embedding(128, d_model // 4)
        self.velocity_emb = nn.Embedding(32, d_model // 8)
        self.time_emb = nn.Embedding(100, d_model // 8)
        self.duration_emb = nn.Embedding(64, d_model // 8)
        self.pedal_emb = nn.Embedding(4, d_model // 8)
        self.input_proj = nn.Linear(
            d_model // 4 + d_model // 8 * 4,  # sum of all field dims
            d_model
        )
        self.pos_embedding = nn.Embedding(max_seq_len, d_model)
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=0.0, batch_first=True
        )
        self.transformer = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        # Factored output heads
        self.pitch_head = nn.Linear(d_model, 128)
        self.velocity_head = nn.Linear(d_model, 32)
        self.time_head = nn.Linear(d_model, 100)
        self.duration_head = nn.Linear(d_model, 64)
        self.pedal_head = nn.Linear(d_model, 4)
        self.brightness_head = nn.Linear(d_model, 16)
        self.attack_head = nn.Linear(d_model, 16)
        self.richness_head = nn.Linear(d_model, 16)
        self.max_seq_len = max_seq_len
    
    def forward(self, pitch, velocity, time_shift, duration, pedal):
        B, T = pitch.shape
        h = torch.cat([
            self.pitch_emb(pitch),
            self.velocity_emb(velocity),
            self.time_emb(time_shift),
            self.duration_emb(duration),
            self.pedal_emb(pedal),
        ], dim=-1)
        h = self.input_proj(h) + self.pos_embedding(torch.arange(T, device=pitch.device).unsqueeze(0))
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=pitch.device)
        memory = torch.zeros(B, 1, self.d_model, device=pitch.device)
        h = self.transformer(h, memory, tgt_mask=mask)
        return {
            'pitch': self.pitch_head(h),
            'velocity': self.velocity_head(h),
            'time': self.time_head(h),
            'duration': self.duration_head(h),
            'pedal': self.pedal_head(h),
            'brightness': self.brightness_head(h),
            'attack': self.attack_head(h),
            'richness': self.richness_head(h),
        }

In [ ]:
# --- Benchmark Function ---

def benchmark_model(model, create_input_fn, name, n_warmup=10, n_runs=100, device='cpu'):
    """Benchmark single-step inference latency."""
    model = model.to(device).eval()
    
    # Count parameters
    n_params = sum(p.numel() for p in model.parameters())
    
    # Warmup
    with torch.no_grad():
        for _ in range(n_warmup):
            inputs = create_input_fn(device)
            if isinstance(inputs, tuple):
                _ = model(*inputs)
            else:
                _ = model(inputs)
    
    if device == 'mps':
        torch.mps.synchronize()
    
    # Benchmark
    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            inputs = create_input_fn(device)
            
            start = time.perf_counter()
            if isinstance(inputs, tuple):
                _ = model(*inputs)
            else:
                _ = model(inputs)
            if device == 'mps':
                torch.mps.synchronize()
            end = time.perf_counter()
            
            latencies.append((end - start) * 1000)  # ms
    
    return {
        'name': name,
        'params': n_params,
        'params_M': n_params / 1e6,
        'mean_ms': np.mean(latencies),
        'median_ms': np.median(latencies),
        'p95_ms': np.percentile(latencies, 95),
        'p99_ms': np.percentile(latencies, 99),
        'min_ms': np.min(latencies),
        'max_ms': np.max(latencies),
    }

In [ ]:
# --- Run Benchmarks ---

# Test different context lengths (how much history the model sees)
context_lengths = [32, 64, 128, 256]

configs = [
    # (name, model, input_fn_factory)
    ('Transformer-Tiny (2L/128d)', 
     lambda: SmallTransformer(d_model=128, nhead=4, num_layers=2),
     lambda ctx_len: lambda dev: torch.randint(0, 512, (1, ctx_len), device=dev)),
    
    ('Transformer-Small (4L/256d)',
     lambda: SmallTransformer(d_model=256, nhead=4, num_layers=4),
     lambda ctx_len: lambda dev: torch.randint(0, 512, (1, ctx_len), device=dev)),
    
    ('Transformer-Medium (6L/384d)',
     lambda: SmallTransformer(d_model=384, nhead=6, num_layers=6),
     lambda ctx_len: lambda dev: torch.randint(0, 512, (1, ctx_len), device=dev)),
    
    ('GRU-Small (2L/256d)',
     lambda: SimpleGRU(d_model=256, num_layers=2),
     lambda ctx_len: lambda dev: torch.randint(0, 512, (1, ctx_len), device=dev)),
    
    ('GRU-Medium (3L/384d)',
     lambda: SimpleGRU(d_model=384, num_layers=3),
     lambda ctx_len: lambda dev: torch.randint(0, 512, (1, ctx_len), device=dev)),
    
    ('CompoundEvent-4L/256d',
     lambda: CompoundEventTransformer(d_model=256, nhead=4, num_layers=4),
     lambda ctx_len: lambda dev: (
         torch.randint(0, 128, (1, ctx_len), device=dev),
         torch.randint(0, 32, (1, ctx_len), device=dev),
         torch.randint(0, 100, (1, ctx_len), device=dev),
         torch.randint(0, 64, (1, ctx_len), device=dev),
         torch.randint(0, 4, (1, ctx_len), device=dev),
     )),
]

results = []
for name, model_fn, input_fn_factory in configs:
    for ctx_len in context_lengths:
        print(f'Benchmarking {name} @ ctx={ctx_len}...')
        model = model_fn()
        input_fn = input_fn_factory(ctx_len)
        result = benchmark_model(model, input_fn, f'{name} (ctx={ctx_len})', device=device)
        result['context_length'] = ctx_len
        result['arch'] = name
        results.append(result)
        print(f'  → median={result["median_ms"]:.2f}ms, p95={result["p95_ms"]:.2f}ms, params={result["params_M"]:.2f}M')

results_df = pd.DataFrame(results)
print('\nDone!')

In [ ]:
# --- Visualize Results ---
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Latency vs context length by architecture
for arch in results_df['arch'].unique():
    subset = results_df[results_df['arch'] == arch]
    axes[0].plot(subset['context_length'], subset['median_ms'], 'o-', label=arch)

axes[0].axhline(y=20, color='red', linestyle='--', alpha=0.7, label='20ms target')
axes[0].axhline(y=50, color='orange', linestyle='--', alpha=0.7, label='50ms acceptable')
axes[0].set_xlabel('Context Length (events)')
axes[0].set_ylabel('Median Latency (ms)')
axes[0].set_title('Inference Latency vs Context Length')
axes[0].legend(fontsize=8)
axes[0].set_yscale('log')

# Params vs latency
ctx128 = results_df[results_df['context_length'] == 128]
axes[1].scatter(ctx128['params_M'], ctx128['median_ms'], s=100, zorder=5)
for _, row in ctx128.iterrows():
    axes[1].annotate(row['arch'], (row['params_M'], row['median_ms']), fontsize=7, ha='left')
axes[1].axhline(y=20, color='red', linestyle='--', alpha=0.7, label='20ms target')
axes[1].set_xlabel('Parameters (M)')
axes[1].set_ylabel('Median Latency (ms) @ ctx=128')
axes[1].set_title('Model Size vs Latency')
axes[1].legend()

plt.tight_layout()
plt.savefig(str(Path.home() / 'Projects' / 'apollo' / 'docs' / 'latency_benchmark.png'), dpi=150)
plt.show()

In [ ]:
# --- Summary Table ---
summary = results_df.pivot_table(
    index='arch', columns='context_length', 
    values='median_ms', aggfunc='first'
).round(2)
print('Median Latency (ms) by Architecture and Context Length:')
print(summary.to_string())

# Identify viable architectures
print('\n=== Viability Assessment (target: <20ms median at ctx=128) ===')
for _, row in results_df[results_df['context_length'] == 128].iterrows():
    viable = '✅' if row['median_ms'] < 20 else ('⚠️' if row['median_ms'] < 50 else '❌')
    print(f'{viable} {row["arch"]}: {row["median_ms"]:.2f}ms (p95={row["p95_ms"]:.2f}ms), {row["params_M"]:.2f}M params')

# Save results
bench_path = Path.home() / 'Projects' / 'apollo' / 'docs' / 'latency_benchmark.json'
results_df.to_json(bench_path, orient='records', indent=2)
print(f'\nResults saved to {bench_path}')